# Notebook 08: Limitation Experiments

**Agency Calculus Empirical Validation — Paper C**

These experiments demonstrate what happens when the JAM objective is
degraded. They validate the theoretical claim that the structural guarantee
depends on the precise form of the objective — any approximation
reopens the compensation window.

**Two degradation variants:**

1. **JAM + ε**: `R = log(min(u_i) + 0.1)`
   - Converts infinite gradient at zero into finite gradient 1/ε = 10
   - Should show floor degradation relative to true JAM

2. **JAM + softmin**: `R = log(softmin(u_i, τ=0.5))`
   - Approximates minimum with smooth function
   - Opens a compensation window of width ~τ
   - Should show floor degradation proportional to τ

**Prediction:** Both variants should produce lower floor utility than true JAM,
demonstrating that the structural guarantee is non-robust to approximation.

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────
# If ai_economist is missing, run notebook 01 first (it handles installation
# and the required kernel restart).
import sys, os

try:
    import ai_economist  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        "\n❌  ai_economist not found. Run notebook 01_setup_and_test first,\n"
        "    restart the kernel, then return here."
    )

# Add src/ to path
for candidate in [
    '/content/ac-validation/src',
    os.path.join(os.getcwd(), '..', 'src'),
    os.path.join(os.getcwd(), 'src'),
]:
    if os.path.exists(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
        print(f'src on path: {candidate}')
        break


In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from training import run_training, TOTAL_TIMESTEPS
from ac_rewards import jam_reward, jam_reward_epsilon, jam_reward_softmin, softmin
from metrics import MetricsLogger
from plotting import plot_limitation_comparison, CONDITION_COLORS

RESULTS_DIR = '../results'
FIGURES_DIR = '../results/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# Limitation experiment uses fewer steps (3M) — 3 seeds each
LIMIT_STEPS = 3_000_000
LIMIT_SEEDS = 3

print('Limitation experiment variants:')
print('  jam_epsilon: R = log(min(u) + 0.1)')
print('  jam_softmin: R = log(softmin(u, tau=0.5))')
print(f'Steps: {LIMIT_STEPS:,}  Seeds: {LIMIT_SEEDS}')

## 1. Theoretical Analysis: What Epsilon Does

In [ ]:
# Show the gradient difference analytically
epsilons = [0.001, 0.01, 0.1, 0.5, 1.0]
x = np.linspace(0.001, 2, 500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Reward curves
ax1.plot(x, np.log(x), 'k-', linewidth=3, label='JAM: log(x) — NO epsilon', zorder=10)
colors_eps = plt.cm.Reds(np.linspace(0.3, 0.9, len(epsilons)))
for eps, color in zip(epsilons, colors_eps):
    ax1.plot(x, np.log(x + eps), color=color, linewidth=1.5, linestyle='--',
             label=f'log(x+{eps})')
ax1.set_ylim(-5, 2)
ax1.set_xlim(0, 2)
ax1.axvline(0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Floor utility min(u_i)')
ax1.set_ylabel('Planner reward')
ax1.set_title('Effect of Epsilon on JAM Reward')
ax1.legend(fontsize=8)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Gradient magnitude
ax2.semilogy(x, 1/x, 'k-', linewidth=3, label='JAM gradient: 1/x → ∞', zorder=10)
for eps, color in zip(epsilons, colors_eps):
    ax2.semilogy(x, 1/(x + eps), color=color, linewidth=1.5, linestyle='--',
                 label=f'1/(x+{eps}) → 1/{eps}')
ax2.set_xlim(0, 2)
ax2.set_xlabel('Floor utility min(u_i)')
ax2.set_ylabel('|∂R/∂floor| (log scale)')
ax2.set_title('Gradient Magnitude: ε Caps the Infinite Penalty')
ax2.legend(fontsize=8)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/epsilon_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

print('Key insight: epsilon converts infinite cost into 1/epsilon cost')
print('With epsilon=0.1: max gradient = 10 (vs infinity for true JAM)')
print('A finite cost means finite compensation is acceptable to the planner')

## 2. Softmin Compensation Window

In [ ]:
# Demonstrate the compensation window opened by softmin
taus = [0.1, 0.5, 1.0, 2.0]

# 4 agents: try transferring from agent 3 to agents 0-2
base = [5.0, 5.0, 5.0, 5.0]
deltas = np.linspace(0, 4.9, 100)

fig, axes = plt.subplots(1, len(taus) + 1, figsize=(18, 4))

# True JAM
jam_rewards = []
for d in deltas:
    u = [5 + d/3, 5 + d/3, 5 + d/3, max(5 - d, 1e-8)]
    jam_rewards.append(jam_reward(u))
axes[0].plot(deltas, jam_rewards, 'k-', linewidth=2)
axes[0].set_title('True JAM: log(min)\n(monotone decreasing)')
axes[0].set_xlabel('Transfer from floor')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Softmin variants
colors_tau = plt.cm.Blues(np.linspace(0.4, 0.9, len(taus)))
for ax, tau, color in zip(axes[1:], taus, colors_tau):
    softmin_rewards = []
    for d in deltas:
        u = [5 + d/3, 5 + d/3, 5 + d/3, max(5 - d, 1e-8)]
        softmin_rewards.append(jam_reward_softmin(u, tau=tau))
    ax.plot(deltas, softmin_rewards, color=color, linewidth=2)
    ax.set_title(f'JAM+softmin τ={tau}\n(has local max at d≈{tau:.1f})')
    ax.set_xlabel('Transfer from floor')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Softmin Opens a Compensation Window Proportional to τ', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/softmin_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Train JAM + Epsilon (Limitation Experiment)

In [ ]:
SEED = 0  # Run seeds 0, 1, 2 separately
CONDITION = 'jam_epsilon'

save_path = f'{RESULTS_DIR}/{CONDITION}_seed{SEED}_metrics.npz'
if os.path.exists(save_path):
    print(f'Already complete: {save_path}')
else:
    print(f'Training {CONDITION} (epsilon=0.1), seed {SEED}')
    print('This SHOULD show floor degradation vs true JAM')
    logger = run_training(
        condition=CONDITION,
        seed=SEED,
        total_timesteps=LIMIT_STEPS,
        results_dir=RESULTS_DIR,
    )
    print(f'Saved to {save_path}')

## 4. Train JAM + Softmin (Limitation Experiment)

In [ ]:
SEED = 0
CONDITION = 'jam_softmin'

save_path = f'{RESULTS_DIR}/{CONDITION}_seed{SEED}_metrics.npz'
if os.path.exists(save_path):
    print(f'Already complete: {save_path}')
else:
    print(f'Training {CONDITION} (tau=0.5), seed {SEED}')
    print('This SHOULD show floor degradation vs true JAM')
    logger = run_training(
        condition=CONDITION,
        seed=SEED,
        total_timesteps=LIMIT_STEPS,
        results_dir=RESULTS_DIR,
    )
    print(f'Saved to {save_path}')

## 5. Compare JAM vs JAM+ε vs JAM+softmin

In [ ]:
from plotting import merge_seed_results

limitation_conditions = ['jam', 'jam_epsilon', 'jam_softmin']
limitation_loggers = {}

for condition in limitation_conditions:
    loggers = []
    for seed in range(LIMIT_SEEDS):
        path = f'{RESULTS_DIR}/{condition}_seed{seed}_metrics.npz'
        if os.path.exists(path):
            loggers.append(MetricsLogger.load(path))
    limitation_loggers[condition] = loggers
    print(f'{condition}: {len(loggers)} seeds loaded')

limit_data = {}
for condition, loggers in limitation_loggers.items():
    if not loggers:
        continue
    steps, mean, std = merge_seed_results(loggers, 'floor_utility_mean')
    if len(steps) > 0:
        limit_data[condition] = {'steps': steps, 'floor_utility': mean, 'floor_utility_std': std}

if limit_data:
    fig = plot_limitation_comparison(
        limit_data,
        metric='floor_utility',
        title='Limitation Experiment: Floor Utility Degradation\nJAM vs JAM+ε vs JAM+softmin',
        save_path=f'{FIGURES_DIR}/limitation_floor_utility.png',
    )
    plt.show()

    print('\nExpected: JAM > JAM+epsilon ≈ JAM+softmin on floor_utility')
    print('Any epsilon or approximation degrades the structural guarantee')
else:
    print('No limitation experiment data yet.')

## 6. Interpretation

These limitation experiments confirm the theoretical claim:

**The structural guarantee is fragile by design.**

Any approximation to the JAM objective — whether epsilon stabilization or soft
minimum — converts the infinite cost into a finite one. A finite cost means
the planner can afford to compress the floor if the payoff is high enough.

This is not a weakness of the approach — it is the point. The guarantee
is not robust to approximation because it relies on the mathematical structure
of log(x) near zero. Paper B makes this explicit.

These experiments provide empirical evidence that the theoretical analysis
of the compensation mechanism is correct.